# 4 &mdash; Uncertainty quantification with a replica-exchange SGLD ensemble

Demonstration of **Section 3.5** and of Table 2 / Figure 5 of
[*B-LSTM-MIONet*](https://arxiv.org/abs/2311.16519), arXiv:2311.16519, on the Lorenz system of
[`01_lorentz.ipynb`](01_lorentz.ipynb).

## What reSGLD does

Instead of a single set of weights, the Bayesian version samples from the posterior
$P(\theta \mid \mathcal{D})$ and predicts with an **$M$-ensemble** $\{\theta_k\}_{k=1}^M$. The
sampler is *replica exchange stochastic gradient Langevin dynamics*: two chains run side by side
on the same mini-batches, each a Langevin diffusion with momentum,

$$v \leftarrow -\eta\,\frac{\nabla L}{\sigma} + (1-\alpha)\,v + \xi,\qquad
  \xi \sim \mathcal{N}\!\left(0, \text{scale}^2 I\right),\qquad
  \text{scale} = \sqrt{2(\alpha-\beta)\eta}\,\sqrt{\tau},$$

with $\sigma = 2\,\text{level}^2$ the assumed noise variance of the targets and
$\beta = \tfrac12 v\,\eta$. The noise $\xi$ is drawn independently for every parameter entry.

* the **exploit** chain is cold (small $\tau$): almost no injected noise, so it settles into the
  mode it is in and refines it. Its parameters are what gets sampled;
* the **explore** chain is hot ($\tau$ twice as large): more noise, larger jumps, so it wanders and
  can leave a local minimum;
* after every epoch the two chains propose a **swap** of their parameters, accepted with
  probability $\min(1, r)$ where
  $r = \exp\!\big[(\tau_{\text{exploit}}^{-1}-\tau_{\text{explore}}^{-1})\,(N\,L_{\text{exploit}} - N\,L_{\text{explore}} - \text{correction})\big]$.
  A swap is likely exactly when the hot chain has stumbled on a *better* region than the cold one,
  which the cold chain then takes over and refines.

Once the burn-in `epochs - (n_ensemble + 1)` is over, the exploit chain contributes one ensemble
member per epoch. Predicting with all of them gives a mean $\mu$ and a standard deviation
$\sigma_{\text{pred}}$; the 95 % band is $\mu \pm 1.96\,\sigma_{\text{pred}}$, and the **PICP** is
the fraction of true values that fall inside it.

In [ ]:
import logging
import os
import warnings
from pathlib import Path

# Locate the repository root, whether the kernel started in notebooks/ or at the root.
ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "lorentz.yaml").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("could not find the repository root (no configs/lorentz.yaml above cwd)")
    ROOT = ROOT.parent

# Paths inside a configuration file are relative to the working directory, so work from the
# repository root: data/, mlruns/ and figures/ are then created there (all three are git-ignored).
os.chdir(ROOT)
os.environ["MLFLOW_TRACKING_URI"] = str(ROOT / "mlruns")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"  # set before mlflow is imported
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

print("repository root     :", ROOT)
print("MLFLOW_TRACKING_URI :", os.environ["MLFLOW_TRACKING_URI"])

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, display

from blstm_mionet.config import load_config
from blstm_mionet.data.datasets import prepare_torch_dataset, split_dataset
from blstm_mionet.data.generate import generate_trajectories, save_dataset
from blstm_mionet.evaluation.evaluate import evaluate_ensemble
from blstm_mionet.evaluation.plotting import ensure_directory, plot_comparison_uq
from blstm_mionet.models import build_model, dataset_preparer_for
from blstm_mionet.training import tracking
from blstm_mionet.training.resgld import train_resgld
from blstm_mionet.training.trainer import train_adam
from blstm_mionet.utils.device import resolve_device
from blstm_mionet.utils.seed import set_seed

# blstm_mionet.evaluation.plotting selects the non-interactive "Agg" backend when it is
# imported, so ask for the inline backend again afterwards.
%matplotlib inline

print("torch               :", torch.__version__)

## Settings: `QUICK` versus the paper

reSGLD costs two forward/backward passes per mini-batch, so it is about twice as expensive per
epoch as Adam. The `QUICK` run therefore does something the paper does not: it **warm starts**
both chains from a short Adam run, so that the few sampling epochs happen near a mode instead of
at the random initialisation. It also raises the temperature $\tau$, because at the paper's
$\tau = 10^{-7}$ the injected noise over eight epochs is far too small to spread the ensemble at
all &mdash; the band would be invisible.

| | `QUICK` | paper, Sec. 3.5 (`configs/bayesian/lorentz.yaml`) |
| --- | --- | --- |
| initialisation | 5 Adam epochs, then reSGLD | reSGLD from scratch |
| reSGLD epochs | 8 | 400 (`--epochs 400` as in `scripts/reproduce_bayesian.sh`; not stated in the paper) |
| `bayesian.n_ensemble` | 5 | 360 |
| burn-in = `epochs - (n_ensemble + 1)` | 2 | 39 |
| members used at inference | all 5 | $M = 300$ (`inference.n_ensemble`) |
| `exploit.tau` / `explore.tau` | $10^{-3}$ / $2\times10^{-3}$ | $10^{-7}$ / $2\times10^{-7}$ |
| `eta`, `alpha`, `v` | $10^{-4}$, 0.1, 0.1 | $10^{-4}$, 0.1, 0.1 |
| `level` ($\sigma = 2\,\text{level}^2$) | 15 | 15 |
| `grad_norm` | 50 | 50 |

Everything else (data, architecture) is the `QUICK` Lorenz setting of notebook 01.

In [ ]:
QUICK = True

QUICK_OVERRIDES = [
    "data.n_sample=40",
    "data.t_max=2.0",
    "data.output=data/lorentz_uq_train.npy",
    "model.branch_state.width=100",
    "model.branch_memory.width=100",
    "model.branch_memory.lstm_size=64",
    "model.branch_memory.lstm_layer_num=1",
    "model.trunk.width=100",
    "training.datafile=data/lorentz_uq_train.npy",
    "training.search_num=40",
    "training.batch_size=64",
    "training.epochs=8",
    "training.device=cpu",
    "training.run_name=lorentz_resgld_quick",
    "training.registered_model_name=null",
    "training.plot_trajs=false",
    "inference.search_num=20",
    "inference.device=cpu",
    "inference.plot_trajs=false",
    "inference.n_ensemble=null",
    # a warmer posterior so that eight epochs of sampling produce a visible band
    "bayesian.n_ensemble=5",
    "bayesian.exploit.tau=1.0e-3",
    "bayesian.explore.tau=2.0e-3",
]

# Paper scale: configs/lorentz.yaml + configs/bayesian/lorentz.yaml as shipped.
FULL_OVERRIDES = [
    "training.device=0",
    "inference.device=0",
]

config = load_config(
    ROOT / "configs" / "lorentz.yaml",
    QUICK_OVERRIDES if QUICK else FULL_OVERRIDES,
    bayesian_path=ROOT / "configs" / "bayesian" / "lorentz.yaml",
)
bayes = config.bayesian

WARM_START_EPOCHS = 5 if QUICK else 0

print(f"sigma = 2 * level^2          : {bayes.sigma}")
print(f"exploit: tau={bayes.exploit.tau:g}, eta={bayes.exploit.eta:g}, "
      f"alpha={bayes.exploit.alpha}, noise scale={bayes.exploit.scale:.3e}")
print(f"explore: tau={bayes.explore.tau:g}, eta={bayes.explore.eta:g}, "
      f"alpha={bayes.explore.alpha}, noise scale={bayes.explore.scale:.3e}")
print(f"1/tau_exploit - 1/tau_explore: {bayes.tau_delta:g}")
print(f"epochs={config.training.epochs}, n_ensemble={bayes.n_ensemble}, "
      f"burn-in after epoch {bayes.burn_in(config.training.epochs)}")

## 1. Data, model and (in `QUICK` mode) an Adam warm start

The dataset and the architecture are the ones of notebook 01; see there for the masking of
Section 3.4.

In [ ]:
set_seed(config.data.seed)
train_raw = generate_trajectories(config.data)
save_dataset(train_raw, config.data.output)

set_seed()
device = resolve_device(config.training.device, verbose=True)
preparer = dataset_preparer_for(config.model.architecture)

train_only, _ = split_dataset(train_raw, test_size=config.training.holdout_size)
train_dataset, _, state_feature_num = prepare_torch_dataset(
    train_only,
    preparer,
    state_component=config.training.state_component,
    search_len=config.training.search_len,
    search_num=config.training.search_num,
    search_random=config.training.search_random,
    offset=config.training.offset,
    t_max=config.training.t_max,
    scale_mode=config.training.scale_mode,
    device=device,
    verbose=False,
)

model = build_model(config.model, state_feature_num)
print(f"{sum(p.numel() for p in model.parameters()):,} trainable parameters "
      f"in {config.model.architecture}")

tracking.configure_tracking(config.tracking.uri)

if WARM_START_EPOCHS:
    warm_cfg = copy.deepcopy(config.training)
    warm_cfg.epochs = WARM_START_EPOCHS
    warm_cfg.save_model = False
    warm_cfg.run_name = f"{config.training.run_name}_warmstart"
    with tracking.start_run(config.training.experiment_name, warm_cfg.run_name):
        warm = train_adam(warm_cfg, model, train_dataset, device)
    print(f"\nwarm start: best val_loss = {warm['best_metric']['val_loss']:.5f}")

## 2. Sample the posterior with reSGLD

`train_resgld` needs an active MLflow run. It logs `ge_exploit`, `ge_explore`, `exchange_rate`,
`swap_probability` and `swap` every epoch, writes each ensemble member as
`ensemble/member_XXXX.pt`, and finally logs the best exploit model under `model` so that the
*class* can be recovered later.

```bash
blstm-mionet train --config configs/lorentz.yaml \
    --bayesian configs/bayesian/lorentz.yaml \
    --data data/lorentz_uq_train.npy --epochs 8 \
    --set bayesian.n_ensemble=5 \
    --set bayesian.exploit.tau=1.0e-3 --set bayesian.explore.tau=2.0e-3 \
    --device cpu --no-plot
# prints "Logged 5 ensemble members to runs:/<id>/ensemble"
```

The command line has no equivalent of the Adam warm start: `train_resgld` always starts the two
chains from the model it is handed, and unlike `train_adam` it does not act on
`training.resume_model`. Run from the CLI, the chains therefore start at the random
initialisation, which is what the paper does &mdash; it just needs many more epochs.

In [ ]:
model_exploit = model
model_explore = copy.deepcopy(model)

with tracking.start_run(config.training.experiment_name, config.training.run_name) as run:
    ENSEMBLE_RUN_ID = run.info.run_id
    tracking.log_config(config)
    resgld_log = train_resgld(
        config=config.training,
        bayesian=bayes,
        model_exploit=model_exploit,
        model_explore=model_explore,
        dataset=train_dataset,
        device=device,
    )

print(f"\ncollected {resgld_log['n_ensemble']} ensemble members")
print(f"best exploit loss / sigma : {resgld_log['best_ge']:.3e}")
print(f"swaps                     : {sum(resgld_log['switches'])} of {config.training.epochs} epochs")
print(f"ensemble artifacts        : runs:/{ENSEMBLE_RUN_ID}/ensemble")

In [ ]:
epochs = np.arange(len(resgld_log["ge exploit"]))
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))

left.plot(epochs, resgld_log["ge exploit"], "o-", color="tab:blue", label="exploit (cold)")
left.plot(epochs, resgld_log["ge explore"], "s-", color="tab:red", label="explore (hot)")
left.axvline(bayes.burn_in(config.training.epochs) + 0.5, color="0.4", ls=":",
             label="end of burn-in")
left.set_yscale("log")
left.set_xlabel("epoch")
left.set_ylabel(r"loss / $\sigma$")
left.set_title("the two Langevin chains")
left.legend(fontsize=8)

right.bar(epochs, np.minimum(1.0, resgld_log["exchange rate"]), color="0.75",
          label="swap probability")
swapped = np.flatnonzero(resgld_log["switches"])
right.plot(swapped, np.ones_like(swapped), "v", color="tab:green", ms=9, label="swap accepted")
right.set_ylim(0, 1.15)
right.set_xlabel("epoch")
right.set_title("replica exchange")
right.legend(fontsize=8)

fig.tight_layout()
plt.show()

## 3. Load the members back and predict with the whole ensemble

`evaluate_ensemble` loads each `member_XXXX.pt` state dict into the model once, runs it over the
whole test set, and reduces the $M$ prediction vectors to a mean, a standard deviation and one
draw from the posterior predictive. It reports the relative $L_1$/$L_2$ errors of the mean and of
that draw, plus the PICP.

```bash
blstm-mionet infer-bayesian --config configs/lorentz.yaml \
    --data data/lorentz_uq_train.npy --run runs:/<run id> --device cpu
```

In [ ]:
inference = config.inference
inference.datafile = config.data.output
inference.run = f"runs:/{ENSEMBLE_RUN_ID}"

_, test_split = split_dataset(train_raw, test_size=1.0, verbose=False)
test_dataset, _, _ = prepare_torch_dataset(
    test_split,
    preparer,
    state_component=inference.state_component,
    search_len=inference.search_len,
    search_num=inference.search_num,
    search_random=inference.search_random,
    offset=inference.offset,
    t_max=inference.t_max,
    scale_mode=inference.scale_mode,
    device=device,
    verbose=False,
)

local_dir = tracking.download_artifacts(ENSEMBLE_RUN_ID, tracking.ENSEMBLE_ARTIFACT_PATH)
member_paths = sorted(Path(local_dir).glob("member_*.pt"))
if inference.n_ensemble is not None:
    member_paths = member_paths[: inference.n_ensemble]
print(f"{len(member_paths)} ensemble members in {local_dir}")

ensemble = evaluate_ensemble(inference, model_exploit, test_dataset, member_paths, device)

## 4. The 95 % confidence band

`plot_comparison_uq` is the function that produced Figure 5 (Lorenz) and Figure 7 (pendulum) of
the paper: the true trajectory, the ensemble mean, one posterior draw, and the shaded
$\mu \pm 1.96\,\sigma_{\text{pred}}$ band.

In [ ]:
IDX = 0
figure_dir = ensure_directory(inference.figure_dir)
fig_path = str(figure_dir / "uq_trajs_notebook.png")

plot_comparison_uq(
    (ensemble["y_true"][IDX], ensemble["mean"][IDX], ensemble["sample"][IDX]),
    ensemble["std"][IDX],
    ("True", "Ensemble mean", "Ensemble sample"),
    color_list=("red", "blue", "black"),
    linestyle_list=("solid", "dashed", "dotted"),
    xlabel="sub-sequence index",
    ylabel="$x(t_n + h)$",
    fig_path=fig_path,
    font_size="12",
)
plt.rcParams["font.size"] = plt.rcParamsDefault["font.size"]
print(f"figure written to {fig_path}")
display(Image(fig_path))

In [ ]:
inside = np.abs(ensemble["y_true"] - ensemble["mean"]) <= 1.96 * ensemble["std"]

print(f"{'':<34}{'this run':>14}{'paper, M = 300':>18}")
print(f"{'ensemble members M':<34}{ensemble['n_members']:>14}{'300':>18}")
print(f"{'L2 of the mean prediction %':<34}{ensemble['L2_error_mat'].mean() * 100:>14.2f}{'5.83':>18}")
print(f"{'L2 of one posterior draw %':<34}{ensemble['L2_error_sample_mat'].mean() * 100:>14.2f}{'-':>18}")
print(f"{'PICP of the 95% band %':<34}{ensemble['picp'] * 100:>14.2f}{'100.00':>18}")
print(f"\nmean posterior st. dev.      : {ensemble['std'].mean():.4f}")
print(f"mean |mean - truth|          : {np.abs(ensemble['y_true'] - ensemble['mean']).mean():.4f}")
print(f"points inside the band       : {int(inside.sum())} of {inside.size}")
print("\nTable 2 of the paper reports L2 = 5.83 % / PICP = 100.00 % for x(t) and")
print("L2 = 8.33 % / PICP = 97.00 % for y(t); Table 4 reports 3.34 % / 100.00 % for the")
print("pendulum. Those come from M = 300 members of a full-length reSGLD run, so the")
print("QUICK numbers above are not comparable - only the machinery is the same.")

## Reproducing the paper

```bash
# the Section 4.1 dataset
blstm-mionet generate --config configs/lorentz.yaml

# reSGLD: 400 epochs; after the burn-in one member per epoch -> 360 members
blstm-mionet train --config configs/lorentz.yaml \
    --bayesian configs/bayesian/lorentz.yaml --epochs 400 --device 0

# Table 2 / Figure 5: evaluate M = 300 of them (inference.n_ensemble)
blstm-mionet infer-bayesian --config configs/lorentz.yaml \
    --data data/lorentz_N_100_h001_T20.npy --run runs:/<run id> --device 0

# the same for the pendulum (Table 4 / Figure 7)
blstm-mionet train --config configs/pendulum.yaml \
    --bayesian configs/bayesian/pendulum.yaml --epochs 400 --device 0
blstm-mionet infer-bayesian --config configs/pendulum.yaml --run runs:/<run id>
```

Published ensemble results:

| | Lorenz $x(t)$ | Lorenz $y(t)$ | pendulum $\theta(t)$ |
| --- | --- | --- | --- |
| $L_2$ of the ensemble mean | 5.83 % (Table 2) | 8.33 % (Table 2) | 3.34 % (Table 4) |
| PICP of the 95 % band | 100.00 % | 97.00 % | 100.00 % |

Figures 5 and 7 of the paper are reproduced in `assets/paper/fig5_lorentz_uq.png` and
`assets/paper/fig7_pendulum_uq.png`.

The chains, the swap probability and the ensemble artifacts of every run are in `mlruns/`:

```bash
MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri mlruns
```

then open <http://127.0.0.1:5000> and look at the `ge_exploit`, `ge_explore` and `swap` metrics.